In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'cxx=g++'
import pytensor

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


In [8]:
pytensor.__version__

'2.37.0'

In [7]:
print(pytensor.config.gcc__cxxflags)

 -Wno-c++11-narrowing -fno-exceptions -fno-unwind-tables -fno-asynchronous-unwind-tables


In [6]:
import pytensor

In [4]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
import pandas as pd
import numpy as np
import statsmodels.api as sm
import miceforest as mf
import pymc as pm
import pytensor
# pytensor.config.cxx = ""

from pulp import LpProblem, LpMaximize, LpVariable, lpSum
from scipy.spatial.distance import cosine
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression

In [8]:
# Load data (adjust paths to your sources folder)
dfs_data = pd.read_csv('historical_dfs_all.csv')
round_data = pd.read_csv('historical_round_all.csv')
upcoming_data = pd.read_csv('upcoming.csv')

# Preprocess dates
round_data['date'] = pd.to_datetime(round_data['date'])
dfs_data['date'] = pd.to_datetime(dfs_data['date'])
current_date = datetime.now()  # Use current time or set to event date
recent_cutoff = current_date - timedelta(days=30)

# Impute NAs in SG columns with mean
# sg_cols = ['sg_total', 'sg_t2g', 'sg_putt', 'sg_ott', 'sg_arg', 'sg_app']
# for col in sg_cols:
#     new_col = str(col)+'_new'
#     round_data[new_col] = round_data[col].fillna(round_data[col].mean())

In [9]:
#clean data
#remove rows with round scoring NA
round_data_cleaned = round_data[~round_data['score'].isna()]

#remove unwanted columns
round_data_cleaned = round_data_cleaned.drop(['great_shots', 'poor_shots'], axis=1)

#add year column as category
round_data_cleaned['year'] = round_data_cleaned['date'].dt.year.astype('category')

#set golfer id as category
round_data_cleaned['dg_id'] = round_data_cleaned['dg_id'].astype('category')

#drop na (can possibly avoid this with imputation, see below)
# round_data_cleaned.dropna(inplace=True)

#reset index
round_data_cleaned.reset_index(drop=True, inplace=True)

round_data_cleaned.isna().sum()

id_round                0
id_event                0
id_course               0
dg_id                   0
round                   0
start_hole              0
teetime                 0
score                   0
pars                    0
birdies                 0
bogies                  0
doubles_or_worse        0
eagles_or_better        0
sg_total                0
sg_t2g              19737
sg_putt             19737
sg_ott              19737
sg_arg              19737
sg_app              19737
scrambling          22711
prox_rgh            25115
prox_fw             22676
gir                 12639
driving_dist        12631
driving_acc         10488
player_name             0
course_name             0
event_name              0
date                    0
year                    0
dtype: int64

# IMPUTATION

In [7]:
#OLD
def impute_sg_data_OLD(round_data):
    """
    Impute missing SG values using raw scoring data and player history.
    Remove rows with NA in raw features first.
    
    Parameters:
    - round_data: DataFrame with columns like dg_id, sg_total, sg_t2g, ..., score, birdies, bogies, etc.
    
    Returns:
    - round_data with imputed SG columns.
    """
    sg_cols = ['sg_total', 'sg_t2g', 'sg_putt', 'sg_ott', 'sg_arg', 'sg_app']
    reg_features = ['birdies', 'bogies', 'score', 'driving_dist', 'gir']
    available_features = [f for f in reg_features if f in round_data.columns]
    
    # Step 0: Remove rows with NA in raw features
    round_data = round_data.dropna(subset=available_features)
    
    # Step 1: Fill with player-specific means (historical average for that golfer)
    for col in sg_cols:
        player_means = round_data.groupby('dg_id')[col].mean()
        # Use .loc to assign without copy warning
        mask = round_data[col].isna()
        round_data.loc[mask, col] = round_data.loc[mask, 'dg_id'].map(player_means)
    
    # Step 2: For remaining NAs, use regression from raw stats
    for col in sg_cols:
        train_data = round_data.dropna(subset=[col])
        if len(train_data) > 0:
            X = train_data[available_features]
            y = train_data[col]
            reg = LinearRegression().fit(X, y)
            
            missing_rows = round_data[round_data[col].isna()].index  # Get indices
            if len(missing_rows) > 0:
                X_missing = round_data.loc[missing_rows, available_features]
                predicted = reg.predict(X_missing)
                # Use .loc with indices for safe assignment
                round_data.loc[missing_rows, col] = predicted
    
    # Step 3: Last resort - fill any remaining NAs with overall mean (~0 for normalized SG)
    for col in sg_cols:
        mean_val = round_data[col].mean()
        round_data.loc[round_data[col].isna(), col] = mean_val
    
    return round_data

def impute_data(round_data):
    cols_to_split = ['id_round', 'id_course', 'start_hole', 'teetime', 'player_name', 'date']
    df_for_imp = round_data_cleaned.drop(cols_to_split, axis=1)
    df_for_imp['id_event'] = df_for_imp['id_event'].astype('category')
    df_for_imp['course_name'] = df_for_imp['course_name'].astype('category')
    df_for_imp['event_name'] = df_for_imp['event_name'].astype('category')
    df_remaining = round_data_cleaned[cols_to_split].copy()

    #impute
    kernel = mf.ImputationKernel(df_for_imp, num_datasets=5, save_all_iterations_data=True, random_state=1842)
    kernel.mice(iterations=5, verbose=False)
    df_imputed = kernel.complete_data(0)  # pick one, or pool later

    df_total = pd.concat([df_imputed, df_remaining], axis=1)

    return df_total
# Usage example
# round_data = pd.read_csv('historical_round_all.csv')
# imputed_round_data = impute_sg_data(round_data)
# imputed_round_data.to_csv('imputed_round_data.csv', index=False)

In [10]:
# category_cols = ['dg_id', 'id_round', 'id_event', 'id_course', 'teetime', 'player_name', 'course_name', 'event_name', 'date']
category_cols = ['id_round', 'teetime', 'player_name', 'course_name', 'event_name', 'date']
golfer_cols = ['scrambling', 'prox_rgh', 'prox_fw', 'gir', 'driving_dist', 'driving_acc']
sg_cols = ['sg_total', 'sg_putt', 'sg_t2g', 'sg_ott', 'sg_arg', 'sg_app']

In [ ]:
#working on this

#DO 2 IMPUTES
    #1. SG VALUES (SG_PUTT, SG_TOTAL, ETC) - CAN'T DO THIS AS DESCRIBED BECAUSE NEED SOME ROUNDS WITH SG DATA
        #group by event
            #check sg na < sg available, else remove from df
        #normalize each sg cat
    #2. REMAINING VALUES (GIR, DRIVING DISTANCE, ETC) - CAN DO THIS BUT DROPPING FOR NOW DUE TO TIME CONSTRAINTS
        #group by golfer

#IF SAMPLE SIZE IS LOW, IMPUTE USING ENTIRE PGA SET  (add Average, Joe to the list?)



#IMPUTE 1
# Loop over events
imputed_dfs = []
for event, group in round_data_cleaned.groupby(['event_name']):
    #split into numeric and categorical df
    
    group_numeric = group.drop(category_cols, axis=1)
    group_numeric = group.drop(sg_cols, axis=1)
    # group_numeric = group_numeric.copy()[['dg_id', 'round', 
    #    'score', 'pars', 'birdies', 'bogies', 'sg_putt']]
    group_remaining = group[category_cols].copy()

    #if na's exist
    if group_numeric.isna().any().any():
        #get count of rows with missing data
        rows_with_na = group_numeric.isna().any(axis=1).sum()

        #if minimum number of historical rounds exists to impute
        if group_numeric.shape[0]-rows_with_na > minnnnn

    else

    # group = group.drop(columns=['dg_id']) 
    #impute
    kernel = mf.ImputationKernel(group_numeric, num_datasets=5, save_all_iterations_data=True, random_state=1842)
    kernel.mice(iterations=5, verbose=False)
    imputed = kernel.complete_data(0)  # pick one, or pool later

#     #restore full columns
#     imputed = imputed.join(group_category)
#     # imputed['dg_id'] = golfer               # restore after

#     #add to df
#     imputed_dfs.append(imputed)
#     break

# df_imputed = pd.concat(imputed_dfs)

# kernel = mf.ImputationKernel(round_data_cleaned_numeric, num_datasets=5, save_all_iterations_data=True, random_state=1842)
# kernel.mice(iterations=5, verbose=False, min_data_in_leaf=10)  # maxit=5
# imputed_data = kernel.complete_data()  # For one dataset, or loop for all


In [ ]:
#impute new data
# df_imputed = impute_data(round_data_cleaned)
# df_imputed.groupby('course_name').mean(numeric_only=True, skipna = False).to_csv('imputed_data.csv')

#write to csv
# df_imputed.to_csv('df_imputed.csv')

In [11]:
#read from csv
df_imputed = pd.read_csv('df_imputed.csv')
df_imputed['date'] = pd.to_datetime(df_imputed['date'])
df_imputed = df_imputed.drop('Unnamed: 0', axis=1)

# COURSE SIMILARITY

#### these can be grouped (See Torrey Pines)

In [12]:
# Baseline Cosine Similarity Matrix
course_agg = df_imputed.groupby('course_name')[sg_cols + ['birdies', 'bogies', 'score', 'driving_dist', 'gir']].mean()
course_agg = course_agg.fillna(course_agg.mean(numeric_only=True))
features = course_agg.values
features_norm = (features - features.mean(axis=0)) / features.std(axis=0)
courses = course_agg.index.tolist()
similarity_matrix_cosine = pd.DataFrame(index=courses, columns=courses)
for i in range(len(courses)):
    for j in range(len(courses)):
        similarity_matrix_cosine.iloc[i, j] = 1 - cosine(features_norm[i], features_norm[j]) if i != j else 1.0
similarity_matrix_cosine.to_csv('course_similarity_matrix_cosine.csv')

# Use mixed matrix for weighting
similarity_matrix = similarity_matrix_cosine

# DFS TOTAL SPLIT TO ROUND

In [13]:
# Apportion dfs_total by round score (revised: inverse to score for better rounds getting more points)
merged = pd.merge(df_imputed, dfs_data, on=['event_name', 'id_event', 'date', 'player_name', 'dg_id'], how='inner')
merged['rounds_played'] = merged.groupby(['dg_id', 'id_event'])['round'].transform('count')
merged['score_inverse'] = 1 / merged['score']  # Lower score = higher proportion
merged['score_prop'] = merged['score_inverse'] / merged.groupby(['dg_id', 'id_event'])['score_inverse'].transform('sum')
merged['pts_per_round'] = merged['total_pts'] * merged['score_prop']

# WEIGHTINGS

In [14]:
# Weightings
exact_courses = ['Torrey Pines GC (South)', 'Torrey Pines Golf Course (South Course)', 'Torrey Pines (North)', 'Torrey Pines (South)']  # For Farmers
# exact_courses = ['Torrey Pines Golf Course (North Course)', 'Torrey Pines Golf Course (South Course)']  # For Farmers
# merged['is_recent'] = (merged['date'] >= recent_cutoff).astype(int)
# merged['recency_weight'] = np.exp(-0.2 * (current_date - merged['date']).dt.days / 365)
# merged['course_weight'] = 0.3
# for course in exact_courses:
#     merged.loc[merged['course_name'] == course, 'course_weight'] = 3.0
# for idx, row in merged.iterrows():
#     if row['course_name'] not in exact_courses:
#         sim_scores = [similarity_matrix.loc[row['course_name'], ec] for ec in exact_courses if ec in similarity_matrix.columns]
#         if sim_scores:
#             merged.at[idx, 'course_weight'] = max(0.3, max(sim_scores))
# merged['total_weight'] = merged['recency_weight'] * (merged['is_recent'] * 5 + 1) * merged['course_weight']

# New Weighting System: Course history as higher base, recency stronger, plus player experience boost
merged['player_experience'] = merged.groupby('dg_id')['round'].transform('count')
merged['experience_weight'] = np.where(merged['player_experience'] > 50, 2.0, 1.0)  # Boost for veterans like Day

merged['course_weight'] = 0.3  # Default
for course in exact_courses:
    merged.loc[merged['course_name'] == course, 'course_weight'] = 4.0  # Higher for exact history
for idx, row in merged.iterrows():
    if row['course_name'] not in exact_courses:
        sim_scores = [similarity_matrix.loc[row['course_name'], ec] for ec in exact_courses if ec in similarity_matrix.columns]
        if sim_scores:
            merged.at[idx, 'course_weight'] = max(0.3, max(sim_scores))

merged['is_recent'] = (merged['date'] >= recent_cutoff).astype(int)
merged['recency_decay'] = np.exp(-0.15 * (current_date - merged['date']).dt.days / 365)  # Slower decay
merged['recency_weight'] = (merged['is_recent'] * 6 + 1) * merged['recency_decay']  # Stronger recent boost

merged['total_weight'] = merged['course_weight'] * merged['recency_weight'] * merged['experience_weight']  # Multiplied for synergy, with experience to favor pros

# MODELS

In [ ]:
# # Models (e.g., WLS as primary)
# X = merged[sg_cols + ['birdies', 'score']]
# X = sm.add_constant(X)
# y = merged['pts_per_round']
# weights = merged['total_weight']
# model_wls = sm.WLS(y, X, weights=weights).fit()
# # model_wls = sm.WLS(y, X).fit()

# # Predict for upcoming (use player means)
# player_means = merged.groupby('dg_id')[sg_cols + ['birdies', 'score']].mean().reset_index()
# upcoming = upcoming.merge(player_means, on='dg_id', how='left').fillna(0)
# X_pred = sm.add_constant(upcoming[sg_cols + ['birdies', 'score']])
# upcoming['proj_per_round'] = model_wls.predict(X_pred)
# upcoming['proj_total'] = upcoming['proj_per_round'] * 4

# # Expanded features including additional stats
# expanded_features = sg_cols + ['birdies', 'score', 'driving_dist', 'gir', 'driving_acc', 'scrambling', 'prox_rgh', 'prox_fw']

# # Weighted Average for player means (for prediction inputs)
# player_means = merged.groupby('dg_id')[expanded_features].apply(
#     lambda g: g.apply(lambda col: np.average(col, weights=merged.loc[g.index, 'total_weight']))
# ).reset_index()

# # Models (e.g., WLS as primary with expanded features)
# X = merged[expanded_features]
# X = sm.add_constant(X)
# y = merged['pts_per_round']
# weights = merged['total_weight']
# model_wls = sm.WLS(y, X, weights=weights).fit()

# # Predict for upcoming (use weighted player means)
# upcoming = upcoming_data.merge(player_means, on='dg_id', how='left').fillna(0)
# X_pred = sm.add_constant(upcoming[expanded_features])
# upcoming['proj_per_round'] = model_wls.predict(X_pred)
# upcoming['proj_total'] = upcoming['proj_per_round'] * 4

In [ ]:
df = merged.dropna(subset=[
    'sg_total', 'sg_ott', 'sg_app', 'sg_arg', 'sg_putt', 
    'total_pts', 'total_weight'
]).copy()

df['total_weight'] = df['total_weight'].clip(lower=1e-6)

player_idx, players = pd.factorize(df['dg_id'])
n_players = len(players)

with pm.Model() as model:
    mu_sg    = pm.Normal('mu_sg',    0, 1.5, shape=5)
    sigma_sg = pm.HalfNormal('sigma_sg', 1.5, shape=5)
    z        = pm.Normal('z', 0, 1, shape=(n_players, 5))
    player_sg = mu_sg + z * sigma_sg

    # Weighted SG
    sg_dist = pm.Normal.dist(mu=player_sg[player_idx], sigma=1.2)
    sg_loglik = pm.logp(sg_dist, df[['sg_total','sg_ott','sg_app','sg_arg','sg_putt']].to_numpy())
    weighted_sg = df['total_weight'].to_numpy()[:, None] * sg_loglik
    pm.Potential('sg_weighted', pm.math.sum(weighted_sg))

    intercept = pm.Normal('intercept', 50, 15)
    w_sg      = pm.Normal('w_sg', 0, 5, shape=5)
    mu_pts    = intercept + pm.math.dot(player_sg[player_idx], w_sg)

    # Weighted fantasy pts
    pts_dist = pm.Normal.dist(mu=mu_pts, sigma=10)
    pts_loglik = pm.logp(pts_dist, df['total_pts'].to_numpy())
    weighted_pts = df['total_weight'].to_numpy() * pts_loglik
    pm.Potential('pts_weighted', pm.math.sum(weighted_pts))

    trace = pm.sample(1000, tune=1500, target_accept=0.9, chains=4)

# Predictions
player_sg_mean = trace.posterior['player_sg'].mean(['chain','draw']).values
w_mean         = trace.posterior['w_sg'].mean(['chain','draw']).values
intercept_mean = float(trace.posterior['intercept'].mean())

exp_pts = intercept_mean + np.dot(player_sg_mean, w_mean)

pred = pd.DataFrame({
    'dg_id': players,
    'exp_fantasy_pts': exp_pts
}).merge(upcoming_data[['dg_id', 'salary', 'player_name']], 
         on='dg_id', how='right')

pred['exp_fantasy_pts'] = pred['exp_fantasy_pts'].fillna(intercept_mean)

print(pred.sort_values('exp_fantasy_pts', ascending=False))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_sg, sigma_sg, z, intercept, w_sg]


c:\Users\tuska\AppData\Local\Programs\Python\Python312\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

# PuLP DFS LINEUP OPTIMIZATION

In [ ]:
dg_rank = pd.read_csv('farmers_insurance_open_preds_ch_model.csv')
df1 = dg_rank[['player_name', 'inv_rank']] #or inv_rank
df1 = df1.rename({'inv_rank':'proj_total'})
df1
# df2 = dfs_data[['dg_id', 'player_name', 'salary']]
# upcoming = df1.merge(df2, on = ['player_name'], how = 'left')

In [69]:
# PuLP Optimization
prob = LpProblem("FanDuel_Opt", LpMaximize)
players = upcoming['dg_id'].tolist()
x = {p: LpVariable(f"select_{p}", cat='Binary') for p in players}
prob += lpSum([x[p] * upcoming.loc[upcoming['dg_id'] == p, 'proj_total'].values[0] for p in players])
prob += lpSum([x[p] for p in players]) == 6
prob += lpSum([x[p] * upcoming.loc[upcoming['dg_id'] == p, 'salary'].values[0] for p in players]) <= 60000
prob.solve()
selected = [p for p in players if x[p].value() == 1]
lineup = upcoming[upcoming['dg_id'].isin(selected)]
print(lineup[['player_name', 'salary', 'proj_total']])
print(f"Total Salary: {lineup['salary'].sum()}, Proj Pts: {lineup['proj_total'].sum()}")

                player_name  salary  proj_total
1             Bauchou, Zach  7300.0   78.354350
6          Brennan, Michael  8400.0   79.162672
24              Ewart, A.J.  7000.0   78.829839
28              Ford, David  7700.0   78.244530
67              Li, Haotong  8800.0   78.704424
143  Yellamaraju, Sudarshan  7200.0   82.402221
Total Salary: 46400.0, Proj Pts: 475.69803567674927


# 

In [72]:
upcoming.sort_values('proj_total', ascending=False).head(10)

,dg_id,salary,player_name,sg_total,sg_putt,sg_t2g,sg_ott,sg_arg,sg_app,birdies,score,driving_dist,gir,driving_acc,scrambling,prox_rgh,prox_fw,proj_per_round,proj_total
143,29925,7200.0,"Yellamaraju, Sudarshan",0.620859,-0.194927,0.555743,0.193357,0.586209,0.592141,4.554063,68.242484,292.603324,0.704308,0.640978,0.613176,33.076953,29.085702,20.600555,82.402221
6,29197,8400.0,"Brennan, Michael",0.083043,0.234021,0.761332,-0.432633,-0.372500,-0.585576,4.428248,69.288016,301.940118,0.717864,0.604405,0.637971,36.751720,29.146853,19.790668,79.162672
24,21883,7000.0,"Ewart, A.J.",0.132319,0.763501,1.255738,-1.231849,-0.568889,-1.507846,4.628317,69.145140,294.760408,0.711843,0.660305,0.647675,49.661843,26.991107,19.707460,78.829839
67,15310,8800.0,"Li, Haotong",0.479397,-0.510312,0.671421,0.439244,0.352119,0.896826,4.509592,69.135919,296.697662,0.731894,0.614346,0.561275,45.887622,31.914356,19.676106,78.704424
1,19846,7300.0,"Bauchou, Zach",0.354209,0.317735,0.415839,-0.370660,-0.340845,0.361009,4.341950,68.866297,293.800092,0.734430,0.599550,0.566504,54.231849,28.221538,19.588587,78.354350
28,31343,7700.0,"Ford, David",0.182747,-0.079889,0.503625,0.255484,0.058950,0.049886,4.339014,69.172903,297.042318,0.694442,0.653057,0.643807,52.189442,29.432474,19.561132,78.244530
89,13764,8100.0,"Parry, John",0.561797,-0.227397,-0.015816,0.539626,0.836703,0.624527,4.226226,68.948445,292.754427,0.711210,0.577899,0.595626,53.985184,33.071801,19.374872,77.499488
41,20954,7100.0,"Hirata, Kensei",-0.108320,-0.202872,-0.037565,0.316900,0.285622,0.292039,3.980862,69.295691,290.469025,0.728305,0.714772,0.657031,58.986040,29.620102,19.240216,76.960863
57,32457,8200.0,"Keefer, Johnny",0.262395,-0.152226,0.172731,0.059760,-0.001593,0.535768,4.293173,69.556063,300.672774,0.736696,0.522363,0.605362,55.252388,29.059055,19.215325,76.861300
12,30620,7100.0,"Chatfield, Davis",-0.420649,-0.372530,0.541105,0.372018,0.242374,-0.070509,4.193689,69.300258,287.014773,0.697193,0.764319,0.644633,64.703531,29.724476,19.212201,76.848802


In [70]:
df = merged[merged['dg_id']==9771].sort_values('total_weight', ascending=False)[['event_name', 'score', 'total_weight']]
# df[df['event_name'].str.contains('Farmer')]
df

,event_name,score,total_weight
87264,The Genesis Invitational,72.0,6.939639
87263,The Genesis Invitational,76.0,6.939639
87266,The Genesis Invitational,72.0,6.939639
87265,The Genesis Invitational,74.0,6.939639
86037,Farmers Insurance Open,74.0,6.877180
...,...,...,...
5105,RBC Heritage,71.0,0.258707
5106,RBC Heritage,69.0,0.258707
1970,AT&T Pebble Beach Pro-Am,75.0,0.244946
1968,AT&T Pebble Beach Pro-Am,64.0,0.244946
